<a href="https://colab.research.google.com/github/JaneCS2025/StrandsAgent/blob/main/StrandsAgent_SetUp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [8]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [9]:
!ollama pull qwen3:0.6b

In [11]:
%%bash
set -euo pipefail

python -m venv --without-pip /content/strands-venv
python -m pip --python /content/strands-venv install "strands-agents[ollama]"
/content/strands-venv/bin/python -c 'from strands.models.ollama import OllamaModel; print("Strands is ready")'

  Using cached strands_agents-1.65.0-py3-none-any.whl.metadata (22 kB)
  Using cached boto3-1.44.3-py3-none-any.whl.metadata (6.6 kB)
  Using cached botocore-1.44.3-py3-none-any.whl.metadata (5.6 kB)
  Using cached mcp-2.3.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_instrumentation_threading-0.66b2-py3-none-any.whl.metadata (2.1 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached ollama-0.6.3-py3-none-any.whl.metadata (7.6 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.19.3-py3-none-any.whl.metadata (1.7 kB)
  Using cached mcp_types-2.3.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached sse_starlette-3.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached opentelemetry_instrumentation-0.66b2-py3-none-any.whl.metadata (7.2 kB)
  Using cached opentelemetry_semantic_conventions-0.66b2-py3-none-any.whl.m

In [16]:
%%bash
/content/strands-venv/bin/python -u - <<'PY'
from strands import Agent
from strands.models.ollama import OllamaModel

model = OllamaModel(
    host="http://127.0.0.1:11434",
    model_id="qwen3:0.6b",
)

agent = Agent(model=model, callback_handler=None)
# print("Calling Qwen on Colab...")
# print(agent("What is RAG?"))
response = agent("What are the key differences between REST and GraphQL APIs?")
print(response)


The key differences between REST and GraphQL APIs are as follows:

1. **Data Structure**:  
   - **REST**: Uses a flat, simple structure for data (e.g., a single endpoint to retrieve a specific resource).  
   - **GraphQL**: Allows clients to define a specific data model, enabling precise control over the data they retrieve.

2. **Query Flexibility**:  
   - **REST**: Typically requires multiple endpoints (GET, POST, PUT, etc.) for different actions, but each request is independent.  
   - **GraphQL**: Enables clients to define their own data structure in a single query, making it more precise and flexible.

3. **Single Endpoint**:  
   - **REST**: Uses a single endpoint to handle a single resource, making it straightforward.  
   - **GraphQL**: Allows clients to define their own data structure and interact with multiple resources in a single request.

4. **Complexity**:  
   - **REST**: Simple to implement and maintain, though less flexible.  
   - **GraphQL**: More complex but offers

bash: line 14: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


In [18]:
!pip install 'strands-agents[web-fetch]'

In [23]:
from strands import Agent, tool
from strands.models.ollama import OllamaModel
from strands.vended_tools import file_editor
from strands.vended_tools.web_fetch import web_fetch

@tool
def query_product_database(query: str) -> str:
    """Query the internal product database for inventory and pricing information.

    Args:
        query: Search query for products (e.g., "wireless headphones", "USB-C hub")
    """
    products = {
        "wireless headphones": "SKU-WH100: Wireless Headphones Pro - $79.99, 142 in stock, 4.5★ rating, launched 2025-03",
        "usb-c hub": "SKU-UC200: USB-C Hub 7-in-1 - $45.00, 89 in stock, 4.2★ rating, launched 2024-11",
        "mechanical keyboard": "SKU-MK300: Mechanical Keyboard RGB - $149.99, 23 in stock, 4.8★ rating, launched 2025-01",
        "noise cancelling": "SKU-NC400: Noise Cancelling Earbuds - $129.99, 67 in stock, 4.6★ rating, launched 2025-05",
    }
    key = query.lower()
    matches = [info for product_key, info in products.items() if product_key in key]
    if matches:
        return "\n".join(matches)
    return f"No products found matching '{query}'. Available: wireless headphones, usb-c hub, mechanical keyboard, noise cancelling"

# Configure local Ollama model
model = OllamaModel(
    host="http://127.0.0.1:11434",
    model_id="qwen3:0.6b",
)

SYSTEM_PROMPT = """You are a product research analyst. You help the team understand
market positioning by comparing competitor pricing with our internal catalog.

When given a research task:
1. Use web_fetch to gather public market data from the web
2. Use query_product_database to check our internal pricing and inventory
3. Write a brief competitive analysis and save it to report.md using file_editor"""

# Pass the local model to the Agent
agent = Agent(
    model=model,
    tools=[web_fetch, file_editor, query_product_database],
    system_prompt=SYSTEM_PROMPT,
)

result = agent("Research what wireless headphones are trending on the market and compare it against our offerings. "
               "Write a short competitive positioning summary and save it to report.md")
print(result)


Tool #1: web_fetch

Tool #2: query_product_database
To proceed, please provide the specific URL you'd like to fetch from the market (e.g., "https://www.example.com/whats-trending-wireless-headphones"). Once you share this, I'll use the web_fetch tool to retrieve the data.  

**Next Steps:**  
1. Use `web_fetch` with your URL to gather market trends.  
2. Use `query_product_database` to check internal pricing and inventory.  
3. Compare and summarize the findings in a competitive positioning report.  

Let me know when you're ready!To proceed, please provide the specific URL you'd like to fetch from the market (e.g., "https://www.example.com/whats-trending-wireless-headphones"). Once you share this, I'll use the web_fetch tool to retrieve the data.  

**Next Steps:**  
1. Use `web_fetch` with your URL to gather market trends.  
2. Use `query_product_database` to check internal pricing and inventory.  
3. Compare and summarize the findings in a competitive positioning report.  

Let me 